# Residual U-Net — Pore Space Segmentation

**Paper reference:**  
Wang et al. (2024), *Comparative Assessment of U-Net-Based Deep Learning Models  
for Segmenting Microfractures and Pore Spaces in Digital Rocks*, SPE-215117-PA.

**Task:** Binary segmentation — **void (pore) vs non-void (solid)** from clean µCT images.

---
### Expected dataset structure (Google Drive)
```
MyDrive/
  pore_data/
    sample_A/
      raw/   ← clean µCT slices  (*.tif / *.png)
      seg/   ← binary masks      (*.tif / *.png)  0=solid, 255=pore
    sample_B/
      raw/
      seg/
    sample_C/
      raw/
      seg/
    sample_D/
      raw/
      seg/
```
---

## 0. Setup & Mount Drive

In [ ]:
# ── Install / verify dependencies ────────────────────────────────────────────
!pip install -q opencv-python-headless tqdm matplotlib

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Konversi `.raw` → Slice 2D

Jalankan cell ini **sekali saja** untuk mengkonversi semua file `.raw` menjadi slice `.png`.
Setelah selesai tidak perlu dijalankan lagi.

**Struktur Drive sebelum konversi:**
```
MyDrive/
  raw_data/
    Bandera Brown 317/
      BanderaBrown_2d25um_grayscale.raw
      BanderaBrown_2d25um_binary.raw
    <sample_lain>/
      ...
```


In [ ]:
import numpy as np
import cv2
from pathlib import Path
from tqdm.notebook import tqdm

# =============================================================================
# EDIT BAGIAN INI
# =============================================================================

# Folder induk yang berisi subfolder tiap sample di Drive
RAW_DATA_ROOT = Path('/content/drive/MyDrive/Dataset TA/317')

# Folder output hasil konversi (akan dibuat otomatis)
OUT_ROOT = Path('/content/drive/MyDrive/Dataset TA/Image Seg Output')

# Dimensi volume: X x Y x Z  (semua sample sama)
VOL_X, VOL_Y, VOL_Z = 1000, 1000, 1000
DTYPE = np.uint8

# Keyword pengenal dalam nama file
GRAYSCALE_KEYWORD = 'grayscale'   # raw uCT  -> raw/
BINARY_KEYWORD    = 'binary'      # mask     -> seg/

# Axis slicing: 0=X, 1=Y, 2=Z(aksial, default paper)
SLICE_AXIS = 2

# =============================================================================


def read_raw_volume(raw_path, shape, dtype):
    vol = np.fromfile(str(raw_path), dtype=dtype)
    expected = shape[0] * shape[1] * shape[2]
    if vol.size != expected:
        raise ValueError(
            f'{raw_path.name}: butuh {expected} voxel '
            f'({shape[0]}x{shape[1]}x{shape[2]}), dapat {vol.size}'
        )
    # Reshape ke (Z, Y, X) — urutan C-order
    return vol.reshape(shape[2], shape[1], shape[0])


def get_slice(vol, i, axis):
    if   axis == 0: return vol[i, :, :]
    elif axis == 1: return vol[:, i, :]
    else:           return vol[:, :, i]


def convert_sample(sample_dir, out_dir):
    raw_files = list(sample_dir.glob('*.raw'))
    if not raw_files:
        print(f'  [SKIP] Tidak ada .raw di {sample_dir.name}')
        return

    gray_file = next((f for f in raw_files if GRAYSCALE_KEYWORD in f.name.lower()), None)
    bin_file  = next((f for f in raw_files if BINARY_KEYWORD    in f.name.lower()), None)

    if gray_file is None or bin_file is None:
        print(f'  [ERROR] {sample_dir.name}: grayscale={gray_file}, binary={bin_file}')
        return

    print(f'\n[{sample_dir.name}]')
    print(f'  Grayscale : {gray_file.name}  ({gray_file.stat().st_size/1e6:.0f} MB)')
    print(f'  Binary    : {bin_file.name}  ({bin_file.stat().st_size/1e6:.0f} MB)')

    shape = (VOL_X, VOL_Y, VOL_Z)

    print('  Membaca volume grayscale ...')
    vol_gray = read_raw_volume(gray_file, shape, DTYPE)
    print('  Membaca volume binary ...')
    vol_bin  = read_raw_volume(bin_file,  shape, DTYPE)

    # Normalkan mask ke 0/255 jika masih 0/1
    unique_vals = np.unique(vol_bin)
    print(f'  Nilai unik mask sebelum normalisasi: {unique_vals}')
    if set(unique_vals.tolist()) <= {0, 1}:
        vol_bin = (vol_bin * 255).astype(np.uint8)
        print('  Mask dikonversi 0/1 -> 0/255')

    raw_out = out_dir / 'raw'
    seg_out = out_dir / 'seg'
    raw_out.mkdir(parents=True, exist_ok=True)
    seg_out.mkdir(parents=True, exist_ok=True)

    n_slices = vol_gray.shape[SLICE_AXIS]
    print(f'  Menyimpan {n_slices} slice (axis={SLICE_AXIS}) ...')

    for i in tqdm(range(n_slices), desc=f'  {sample_dir.name}', leave=False):
        fname = f'slice_{i:04d}.png'
        cv2.imwrite(str(raw_out / fname), get_slice(vol_gray, i, SLICE_AXIS))
        cv2.imwrite(str(seg_out / fname), get_slice(vol_bin,  i, SLICE_AXIS))

    print(f'  Selesai -> {out_dir}')


# ── Jalankan untuk semua sample ──────────────────────────────────────────────
sample_dirs = sorted([d for d in RAW_DATA_ROOT.iterdir() if d.is_dir()])

if not sample_dirs:
    raise FileNotFoundError(f'Tidak ada subfolder di {RAW_DATA_ROOT}')

print(f'Ditemukan {len(sample_dirs)} sample: {[d.name for d in sample_dirs]}')
print(f'Output -> {OUT_ROOT}')

for sample_dir in sample_dirs:
    sample_name = sample_dir.name.replace(' ', '_')
    convert_sample(sample_dir, OUT_ROOT / sample_name)

print('\nSemua sample selesai!')
print(f'Isi DATA_DIR di cell Config dengan: {OUT_ROOT}')


## 2. Configuration
**Edit only this cell** to match your setup.

In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/content/drive/MyDrive/Dataset TA/Image Seg Output')   # hasil konversi dari cell di atas   # root data folder
OUTPUT_DIR = Path('/content/drive/MyDrive/Dataset TA/Image Seg Output/pore_output') # checkpoints & results
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters (adopted from paper) ─────────────────────────────
PATCH_SIZE   = 256      # crop size per slice; paper uses 512 (reduce if OOM)
BATCH_SIZE   = 4        # paper trains on 2× RTX 3090; reduce for T4
EPOCHS       = 50       # paper: 50 epochs
LR           = 1e-6     # paper: Adam lr = 1e-6
TRAIN_RATIO  = 0.75     # 3 of 4 samples for training; 1 held-out for val
                        # (paper: 90/10 split by sample — adapted for 4 samples)
NUM_WORKERS  = 2        # Colab: 2 is safe
N_STACK      = 5        # 2.5D: central slice ± 2 neighbours (paper default)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config OK — device: {DEVICE}')

## 3. Dataset

In [ ]:
import random
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SUPPORTED = {'.tif', '.tiff', '.png'}

def load_slice_paths(folder: Path) -> list:
    """Return sorted list of image paths inside folder."""
    paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in SUPPORTED)
    if not paths:
        raise FileNotFoundError(f'No slices found in {folder}')
    return paths

def read_gray(path: Path) -> np.ndarray:
    """Load image as uint8 grayscale."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f'Cannot read {path}')
    return img

def to_binary_mask(mask: np.ndarray) -> np.ndarray:
    """Convert mask to {0=solid, 1=pore}. Handles 0/255 or 0/1 encoding."""
    return (mask > 127).astype(np.uint8)


class PoreDataset(Dataset):
    """
    2.5D Pore Segmentation Dataset.

    Input  : (N_STACK, H, W) float32, normalised [0, 1]
             Stack = [central-2, central-1, central, central+1, central+2]
    Target : (H, W) int64, {0=solid, 1=pore}

    Adopted from paper §Methodology:
      - 2.5D strategy: 5 stacked 2D slices as input
      - Random crop + horizontal/vertical flip augmentation during training
      - No synthesis noise (clean µCT workflow)
    """

    def __init__(self, sample_dirs: list, patch_size: int = 256, augment: bool = True):
        self.patch_size = patch_size
        self.augment    = augment

        # Build flat index: list of (raw_paths, seg_paths, slice_idx)
        self.records = []
        for raw_dir, seg_dir in sample_dirs:
            raws = load_slice_paths(raw_dir)
            segs = load_slice_paths(seg_dir)
            if len(raws) != len(segs):
                raise ValueError(
                    f'Slice count mismatch: {raw_dir} has {len(raws)} raw '
                    f'but {len(segs)} seg slices'
                )
            for i in range(len(raws)):
                self.records.append((raws, segs, i))

        print(f'  Dataset: {len(self.records)} slices from {len(sample_dirs)} samples')

    def __len__(self):
        return len(self.records)

    def _get_raw(self, raws, idx):
        idx = max(0, min(idx, len(raws) - 1))   # clamp boundary
        return read_gray(raws[idx]).astype(np.float32)

    def __getitem__(self, idx):
        raws, segs, center = self.records[idx]

        # ── 2.5D stack ──────────────────────────────────────────────────────
        stack = np.stack([
            self._get_raw(raws, center + offset)
            for offset in [-2, -1, 0, 1, 2]
        ], axis=0)                                    # (5, H, W)

        # ── Normalise to [0, 1] ──────────────────────────────────────────────
        stack = stack / 255.0

        # ── Ground truth ─────────────────────────────────────────────────────
        mask = to_binary_mask(read_gray(segs[center]))  # (H, W)

        # ── Random crop ──────────────────────────────────────────────────────
        H, W = mask.shape
        ps   = self.patch_size
        if H > ps and W > ps:
            if self.augment:
                y0 = random.randint(0, H - ps)
                x0 = random.randint(0, W - ps)
            else:
                y0 = (H - ps) // 2   # centre crop for val
                x0 = (W - ps) // 2
            stack = stack[:, y0:y0+ps, x0:x0+ps]
            mask  = mask[y0:y0+ps, x0:x0+ps]
        elif H < ps or W < ps:
            # Reflect-pad smaller images
            ph = max(0, ps - H)
            pw = max(0, ps - W)
            stack = np.pad(stack, ((0,0),(0,ph),(0,pw)), mode='reflect')
            mask  = np.pad(mask,  ((0,ph),(0,pw)),       mode='reflect')

        # ── Random flip augmentation ─────────────────────────────────────────
        if self.augment:
            if random.random() > 0.5:                  # horizontal flip
                stack = np.flip(stack, axis=2).copy()
                mask  = np.flip(mask,  axis=1).copy()
            if random.random() > 0.5:                  # vertical flip
                stack = np.flip(stack, axis=1).copy()
                mask  = np.flip(mask,  axis=0).copy()

        return (
            torch.from_numpy(stack).float(),
            torch.from_numpy(mask.astype(np.int64)),
        )

In [ ]:
# ── Discover samples and split by sample (not by slice) ──────────────────────
# Paper §Model Training: "split into training and testing data sets based on
# rock type to ensure the variations and representativeness"

all_samples = sorted([
    d for d in DATA_DIR.iterdir()
    if d.is_dir() and (d/'raw').exists() and (d/'seg').exists()
])

if not all_samples:
    raise FileNotFoundError(
        f'No samples found in {DATA_DIR}.\n'
        'Each sample folder must contain raw/ and seg/ subdirectories.'
    )

print(f'Found {len(all_samples)} samples: {[s.name for s in all_samples]}')

n_train = max(1, int(len(all_samples) * TRAIN_RATIO))
train_samples = all_samples[:n_train]
val_samples   = all_samples[n_train:] if n_train < len(all_samples) else all_samples[-1:]

print(f'Train : {[s.name for s in train_samples]}')
print(f'Val   : {[s.name for s in val_samples]}')

def make_dir_pairs(samples):
    return [(s/'raw', s/'seg') for s in samples]

train_ds = PoreDataset(make_dir_pairs(train_samples), patch_size=PATCH_SIZE, augment=True)
val_ds   = PoreDataset(make_dir_pairs(val_samples),   patch_size=PATCH_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ── Quick sanity check ────────────────────────────────────────────────────────
imgs, masks = next(iter(train_loader))
print(f'\nBatch shapes → imgs: {imgs.shape}  masks: {masks.shape}')
print(f'Img  range : [{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'Mask unique: {masks.unique().tolist()}')

## 4. Residual U-Net Model
Architecture from paper Fig. 3 — residual blocks in both encoder and decoder.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):
    """
    Two 3×3 conv layers with BatchNorm + ReLU.
    Identity shortcut via 1×1 conv when channel dims differ (paper Fig. 3).
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        # 1×1 shortcut to match channels (paper: I(x) branch)
        self.shortcut = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                          nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class EncoderBlock(nn.Module):
    """ResidualBlock → feature map (skip) + MaxPool (downsampled)."""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch)
        self.pool  = nn.MaxPool2d(2, 2)

    def forward(self, x):
        skip = self.block(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    """Bilinear upsample → concat skip connection → ResidualBlock."""
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.block = ResidualBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.block(torch.cat([x, skip], dim=1))


class ResidualUNet(nn.Module):
    """
    Residual U-Net (Wang et al. 2024, SPE-215117-PA).

    Filters : [32, 64, 128, 256, 512]  — paper Table 6 uniform filter setting
    Input   : (B, 5, H, W)             — 2.5D stack
    Output  : (B, 2, H, W)             — logits for {solid, pore}
    """
    FILTERS = [32, 64, 128, 256, 512]

    def __init__(self, in_ch: int = 5, n_classes: int = 2):
        super().__init__()
        f = self.FILTERS

        # Encoder
        self.enc1 = EncoderBlock(in_ch, f[0])
        self.enc2 = EncoderBlock(f[0],  f[1])
        self.enc3 = EncoderBlock(f[1],  f[2])
        self.enc4 = EncoderBlock(f[2],  f[3])

        # Bottleneck
        self.bottleneck = ResidualBlock(f[3], f[4])

        # Decoder
        self.dec4 = DecoderBlock(f[4], f[3], f[3])
        self.dec3 = DecoderBlock(f[3], f[2], f[2])
        self.dec2 = DecoderBlock(f[2], f[1], f[1])
        self.dec1 = DecoderBlock(f[1], f[0], f[0])

        # Output
        self.out = nn.Conv2d(f[0], n_classes, kernel_size=1)

    def forward(self, x):
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.out(x)


model = ResidualUNet(in_ch=N_STACK, n_classes=2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Residual U-Net | trainable params: {n_params:,}')
print(f'Paper reference (Residual U-Net):   8,295,907 params')

## 5. Metrics
Pixel-wise F1 (overall & pore) + boundary-scaled accuracy — from paper §Evaluation.

In [ ]:
class SegMetrics:
    """
    Accumulates TP/FP/FN/TN over batches.

    Paper metrics (Table 3):
      - F1 Overall  : pixel accuracy across all classes
      - F1 Pore     : harmonic mean of precision & recall for pore class
      - Boundary Acc: fraction of boundary pixels correctly predicted (Eq. 4)
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0
        self.tp_b = self.total_b = 0

    @staticmethod
    def _boundary(mask: torch.Tensor) -> torch.Tensor:
        """2-pixel wide boundary map via morphological dilation - erosion."""
        m = mask.float().unsqueeze(0).unsqueeze(0)   # (1,1,H,W)
        k = torch.ones(1, 1, 3, 3, device=mask.device)
        dilated = (F.conv2d(m, k, padding=1).squeeze() > 0)
        eroded  = (F.conv2d(m, k, padding=1).squeeze() >= 9)
        return dilated & ~eroded

    def update(self, logits: torch.Tensor, targets: torch.Tensor):
        """logits: (B,2,H,W)   targets: (B,H,W) int64"""
        preds = logits.argmax(dim=1)
        for b in range(targets.shape[0]):
            p, t = preds[b], targets[b]
            self.tp += int(((p==1)&(t==1)).sum())
            self.fp += int(((p==1)&(t==0)).sum())
            self.fn += int(((p==0)&(t==1)).sum())
            self.tn += int(((p==0)&(t==0)).sum())
            bnd = self._boundary(t)
            self.tp_b    += int((p[bnd] == t[bnd]).sum())
            self.total_b += int(bnd.sum())

    @property
    def f1_overall(self):
        return (self.tp + self.tn) / (self.tp+self.fp+self.fn+self.tn + 1e-8)

    @property
    def f1_pore(self):
        prec = self.tp / (self.tp + self.fp + 1e-8)
        rec  = self.tp / (self.tp + self.fn + 1e-8)
        return 2*prec*rec / (prec + rec + 1e-8)

    @property
    def boundary_acc(self):
        return self.tp_b / (self.total_b + 1e-8)

    def result(self) -> dict:
        return dict(
            f1_overall   = round(self.f1_overall,  4),
            f1_pore      = round(self.f1_pore,     4),
            boundary_acc = round(self.boundary_acc, 4),
        )

print('SegMetrics class defined.')

## 6. Training Loop

In [ ]:
import time, json
from tqdm.notebook import tqdm

# ── Loss & Optimiser  (Paper: BCE loss + Adam lr=1e-6) ────────────────────────
criterion = nn.CrossEntropyLoss()                    # = multi-class BCE
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = dict(train_loss=[], val_loss=[],
               f1_overall=[], f1_pore=[], boundary_acc=[])
best_f1_pore = 0.0


def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    metrics = SegMetrics()
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, masks)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            metrics.update(logits.detach(), masks)
    return total_loss / len(loader.dataset), metrics.result()


print(f'Starting training for {EPOCHS} epochs …\n')
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, _       = run_epoch(train_loader, train=True)
    val_loss,   val_m   = run_epoch(val_loader,   train=False)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['f1_overall'].append(val_m['f1_overall'])
    history['f1_pore'].append(val_m['f1_pore'])
    history['boundary_acc'].append(val_m['boundary_acc'])

    elapsed = time.time() - t0
    print(
        f'Epoch {epoch:3d}/{EPOCHS} | {elapsed:5.1f}s | '
        f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f} | '
        f'F1_overall={val_m["f1_overall"]:.4f}  '
        f'F1_pore={val_m["f1_pore"]:.4f}  '
        f'Bnd_acc={val_m["boundary_acc"]:.4f}'
    )

    # Save best checkpoint
    if val_m['f1_pore'] > best_f1_pore:
        best_f1_pore = val_m['f1_pore']
        torch.save(
            {'epoch': epoch, 'state_dict': model.state_dict(),
             'optimizer': optimizer.state_dict(), 'val_metrics': val_m},
            OUTPUT_DIR / 'best_model.pth'
        )
        print(f'  ✓ Best model saved  (F1_pore={best_f1_pore:.4f})')

torch.save({'epoch': EPOCHS, 'state_dict': model.state_dict()},
           OUTPUT_DIR / 'final_model.pth')

# Save history
with open(OUTPUT_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

total_h = (time.time() - t_start) / 3600
print(f'\nTraining done in {total_h:.2f} h  |  Best F1_pore = {best_f1_pore:.4f}')

## 7. Training Curves

In [ ]:
import time, json

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], label='train')
axes[0].plot(ep, history['val_loss'],   label='val')
axes[0].set_title('BCE Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=.3)

axes[1].plot(ep, history['f1_overall'], label='F1 Overall')
axes[1].plot(ep, history['f1_pore'],    label='F1 Pore')
axes[1].set_title('F1 Score'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=.3)

axes[2].plot(ep, history['boundary_acc'], color='purple', label='Boundary Acc')
axes[2].set_title('Boundary-Scaled Accuracy'); axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1); axes[2].legend(); axes[2].grid(True, alpha=.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()
print(f'Plot saved → {OUTPUT_DIR}/training_curves.png')

## 8. Inference & Visualisation

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
ckpt = torch.load(OUTPUT_DIR / 'best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'Best model loaded (epoch {ckpt["epoch"]}, val_metrics={ckpt["val_metrics"]})')


def predict_sample(raw_dir: Path, threshold: float = 0.5):
    """
    Run 2.5D inference on all slices in raw_dir.
    Returns list of (H, W) uint8 binary masks {0=solid, 1=pore}.
    """
    raw_paths = load_slice_paths(raw_dir)
    pred_masks, infer_ms = [], []
    n = len(raw_paths)

    for i, _ in enumerate(raw_paths):
        # Build 2.5D stack
        stack = np.stack([
            read_gray(raw_paths[max(0, min(i + off, n-1))]).astype(np.float32)
            for off in [-2, -1, 0, 1, 2]
        ], axis=0) / 255.0                          # (5, H, W)

        t = torch.from_numpy(stack).float().unsqueeze(0).to(DEVICE)  # (1,5,H,W)

        t0 = time.perf_counter()
        with torch.no_grad():
            logits = model(t)
            prob   = F.softmax(logits, dim=1)[0, 1].cpu().numpy()  # pore prob
        infer_ms.append((time.perf_counter() - t0) * 1000)

        pred_masks.append((prob >= threshold).astype(np.uint8))

    print(f'Predicted {n} slices | avg {np.mean(infer_ms):.1f} ms/slice')
    return pred_masks, raw_paths


# ── Run on validation sample ──────────────────────────────────────────────────
# Change to any sample directory you want to evaluate
INFER_SAMPLE = val_samples[0]
print(f'\nRunning inference on: {INFER_SAMPLE.name}')
pred_masks, raw_paths = predict_sample(INFER_SAMPLE / 'raw')

In [ ]:
# ── Evaluate against ground truth ─────────────────────────────────────────────
gt_paths = load_slice_paths(INFER_SAMPLE / 'seg')

eval_metrics = SegMetrics()
for pred, gt_path in zip(pred_masks, gt_paths):
    gt = to_binary_mask(read_gray(gt_path))
    # Wrap as logits shape (1, 2, H, W) for SegMetrics.update()
    H, W = pred.shape
    logit_fake = torch.zeros(1, 2, H, W)
    logit_fake[0, 1][pred == 1] = 1.0
    logit_fake[0, 0][pred == 0] = 1.0
    eval_metrics.update(logit_fake, torch.from_numpy(gt.astype(np.int64)).unsqueeze(0))

print('\n── Evaluation Results ──────────────────────────────')
for k, v in eval_metrics.result().items():
    print(f'  {k:<20s}: {v:.4f}')
print('────────────────────────────────────────────────────')

In [ ]:
# ── Visualise: pick a few representative slices ───────────────────────────────
N_SHOW  = 4
indices = np.linspace(0, len(pred_masks)-1, N_SHOW, dtype=int)

fig, axes = plt.subplots(N_SHOW, 3, figsize=(12, 4*N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(indices):
    raw  = read_gray(raw_paths[idx])
    pred = pred_masks[idx]
    gt   = to_binary_mask(read_gray(gt_paths[idx]))
    err  = (pred != gt).astype(np.uint8)

    axes[row, 0].imshow(raw, cmap='gray')
    axes[row, 0].set_title(f'Raw µCT  (slice {idx})')
    axes[row, 0].axis('off')

    # Overlay prediction on raw image
    overlay = cv2.cvtColor(raw, cv2.COLOR_GRAY2RGB)
    overlay[pred == 1] = [255, 165, 0]          # orange = predicted pore
    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title('Prediction (orange=pore)')
    axes[row, 1].axis('off')

    # Error map: red = false positive, blue = false negative
    err_rgb = np.zeros((*raw.shape, 3), dtype=np.uint8)
    fp_mask = (pred == 1) & (gt == 0)
    fn_mask = (pred == 0) & (gt == 1)
    tp_mask = (pred == 1) & (gt == 1)
    err_rgb[fp_mask] = [255,   0,   0]   # red   = false positive
    err_rgb[fn_mask] = [  0,   0, 255]   # blue  = false negative
    err_rgb[tp_mask] = [  0, 200,   0]   # green = true positive
    axes[row, 2].imshow(err_rgb)
    axes[row, 2].set_title('TP=green  FP=red  FN=blue')
    axes[row, 2].axis('off')

plt.suptitle(f'Sample: {INFER_SAMPLE.name}', fontsize=14, y=1.01)
plt.tight_layout()
out_vis = OUTPUT_DIR / f'vis_{INFER_SAMPLE.name}.png'
plt.savefig(out_vis, dpi=150, bbox_inches='tight')
plt.show()
print(f'Visualisation saved → {out_vis}')

In [ ]:
# ── Save prediction masks to Drive ───────────────────────────────────────────
mask_out = OUTPUT_DIR / f'pred_masks_{INFER_SAMPLE.name}'
mask_out.mkdir(exist_ok=True)

for raw_path, pred in zip(raw_paths, pred_masks):
    cv2.imwrite(str(mask_out / raw_path.name), (pred * 255).astype(np.uint8))

print(f'Saved {len(pred_masks)} binary masks → {mask_out}')

training history plot

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# Load history dari Drive
hist_path = OUTPUT_DIR / 'history.json'

with open(hist_path) as f:
    history = json.load(f)

print(f'History tersedia: {len(history["train_loss"])} epoch')

# Plot
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ep, history['train_loss'], label='train')
axes[0].plot(ep, history['val_loss'],   label='val')
axes[0].set_title('BCE Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=.3)

axes[1].plot(ep, history['f1_overall'], label='F1 Overall')
axes[1].plot(ep, history['f1_pore'],    label='F1 Pore')
axes[1].set_title('F1 Score'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=.3)

axes[2].plot(ep, history['boundary_acc'], color='purple')
axes[2].set_title('Boundary-Scaled Accuracy'); axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1); axes[2].grid(True, alpha=.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()